# Forest Mask CLI Test

`services/forest_mask/cli.py` の `main` をプロジェクトルートから実行する想定でテストする。

- 入力: `data/input/jpg/sample/sample_tree.jpg`
- 出力:
  - `data/intermediate/exg/exg_<timestamp>.jpg`
  - `data/intermediate/forest_mask/fmask_<timestamp>.jpg`

In [ ]:
# プロジェクトルートで実行できるようにカレントを移動
import os
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
    os.chdir(PROJECT_ROOT)

print("cwd:", Path.cwd())

In [ ]:
# モジュールリロードを有効化（実装変更を即反映するため）
%load_ext autoreload
%autoreload 2

In [ ]:
from click.testing import CliRunner

from services.forest_mask.cli import main

INPUT_PATH = "data/input/jpg/sample/sample_tree.jpg"
THRESHOLD = 10.0

runner = CliRunner()
result = runner.invoke(
    main,
    [INPUT_PATH, "--threshold", str(THRESHOLD)],
    catch_exceptions=False,
)

print(result.output)
assert result.exit_code == 0, f"exit_code={result.exit_code}"

In [ ]:
# 出力結果の確認: 入力・ExG・Forest Mask を並べて表示
import cv2
import matplotlib.pyplot as plt

from services.common.image import load_rgb

exg_dir = Path("data/intermediate/exg")
mask_dir = Path("data/intermediate/forest_mask")

latest_exg = max(exg_dir.glob("exg_*.jpg"), key=lambda p: p.stat().st_mtime)
latest_mask = max(mask_dir.glob("fmask_*.jpg"), key=lambda p: p.stat().st_mtime)
print("latest exg :", latest_exg)
print("latest mask:", latest_mask)

rgb = load_rgb(Path(INPUT_PATH))
exg = cv2.imread(str(latest_exg), cv2.IMREAD_GRAYSCALE)
mask = cv2.imread(str(latest_mask), cv2.IMREAD_GRAYSCALE)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
axes[0].imshow(rgb)
axes[0].set_title("input (RGB)")
axes[0].axis("off")
axes[1].imshow(exg, cmap="gray")
axes[1].set_title("ExG")
axes[1].axis("off")
axes[2].imshow(mask, cmap="gray")
axes[2].set_title(f"Forest Mask (threshold={THRESHOLD})")
axes[2].axis("off")
plt.tight_layout()
plt.show()